## Outline

The purpose of this script is to plot group-averaged power maps, PSDs, and coherence maps for each mode inferred by DyNeMo, and save figures to disk

In [ ]:
# Import packages
import os
import pandas as pd
import numpy as np
import glob
import datetime
from matplotlib import pyplot as plt
import yaml
from nilearn import image, datasets
from plotting_functions import *  # custom functions for plotting modes

In [ ]:
# Write out data?
writeOK=True

In [18]:
# Read config file and define paths
with open('0_config.yaml', 'r') as file:
    config = yaml.safe_load(file)
    
subjects_dir = config['dirs']['recon_dir']
proc_dir = config['dirs']['proc_dir']
atlas_dir = config['dirs']['atlas_dir']
model_dir = config['dirs']['model_dir']
results_dir = config['dirs']['results_dir']



# Plotting params
colours = config['misc']['colours']
n_modes = 6

# Other params
run = '3'

# Update model dir according to run #
model_dir = f'{model_dir}_run{run}'

# Make directory for figures
fig_dir = os.path.join(model_dir, "figures")
if writeOK:
    os.makedirs(fig_dir, exist_ok=True)

In [19]:
# Define a helper function to pull the most recent version of a file
# Note that this sleects file based on the date & time that they were last
# modified, not the actual datetime printed in the filename 
def recent_fname(path):
    
    # Takes a partial file path
    files = glob.glob(path)
    most_recent_file = max(files, key=os.path.getmtime)
    
    return most_recent_file

In [20]:
# Files to load
subjects_fname = recent_fname(os.path.join(results_dir, 'subject_order_*.txt')) # contains subjects who went into model training, in the correct order
atlas_fname = os.path.join(atlas_dir, 'MEG_atlas_38_regions_4D.nii.gz')

# Load list of subjects that went into DyNeMo
subjects = np.loadtxt(subjects_fname, dtype=str).tolist()

# Load atlas
atlas = image.load_img(atlas_fname)
coords = np.load(os.path.join(atlas_dir, 'MEG_atlas_38_regions_coords.npy'))
names = np.squeeze(pd.read_csv(os.path.join(atlas_dir, 'MEG_atlas_38_regions_names.csv'), header=None).to_numpy())

# Load DyNeMo outputs
alpha = np.load(os.path.join(model_dir, 'inf_params', 'alp_rw.pkl'),allow_pickle=True)
covs = np.load(os.path.join(model_dir, 'inf_params', 'covs.npy'))
psd = np.load(os.path.join(model_dir, 'spectra', 'psd_rs.npy'))
psd_base = psd[:,1]
psd_coef = psd[:,0]
coh = np.load(os.path.join(model_dir, 'spectra', 'coh.npy'))
freqs = np.load(os.path.join(model_dir, 'spectra', 'f.npy'))

Plotting

In [16]:
# Plot mode PSDs
for mode in range(n_modes):
    mode_power = np.mean(psd_coef[:,mode,:,:], (0,-1))
    
    max_reg = np.argmax(np.mean(psd_coef[:,mode,:,:], (0,-1)))
    base_mean = np.mean(psd_base[:,mode,:,:], 0)
    coef_mean = np.mean(psd_coef[:,mode,:,:], 0)
    mode_psd = (base_mean.T + (coef_mean.T)).T
    
    fig, ax = plt.subplots(figsize=(3.2,2))
    ax.plot(freqs[3:], np.average(mode_psd[:,3:], 0, weights=mode_power), color=colours[mode], linewidth=2, alpha=1, label='Mode PSD')
    ax.plot(freqs[3:], np.mean(base_mean,0)[3:], color='black', linestyle='--', linewidth=2, label='Static PSD')
    
    ax.set_xlabel('Frequency (Hz)', fontsize=12)
    ax.set_ylabel('Amplitude', fontsize=12)
    ax.set_xlim([4, 40])
    #ax.set_ylim([0,0.15])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.legend(frameon=False)
    #ax.set_title(f'Mode {mode+1}')
    plt.tight_layout()
    fig.savefig(os.path.join(model_dir, 'figures', 'mode' + str(mode+1) + '_PSD.png'))
    plt.close()

In [21]:
# Plot mode power maps
for mode in range(n_modes):
    coef_mean = np.mean(psd_coef[:,mode,:,:], 0)
    power = np.mean(coef_mean, -1)
    power *= 100
    
    #img = make_atlas_nifti(atlas, power)
    img = make_4d_atlas_nifti(atlas, power)
    fig = surface_brain_plot(img, subjects_dir, 'pial', 'RdBu_r', threshold=0, cbar_label='Coefficient', figsize=(5,5))
    #plt.title(mode_names[mode])
    fig.savefig(os.path.join(model_dir, 'figures', 'mode' + str(mode+1) + '_power.png'))
    
    plt.close()

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [ ]:
# Plot coherence in (in the top X percent of edges
top_coh = 3 # percent
thresh = 100-top_coh

for mode in range(n_modes):
    mode_coh = np.mean(coh[:,mode,:,:,:], 0)
    freq_range = (freqs>=4) * (freqs<=30)
    conn = np.nanmean(mode_coh[:,:,freq_range], -1)
    triu = np.triu_indices(len(names), k=1)
    np.fill_diagonal(conn, np.nan)
    threshold = np.nanpercentile(np.abs(conn[triu]), thresh)
    fig = glass_brain_plot(conn, coords, threshold, 'Mode Coherence')
    fig.savefig(os.path.join(model_dir, 'figures', 'mode' + str(mode+1) + '_coh.png'))
    plt.close()